# PubMed RAG



## Executive Summary

This notebook demonstrates a complete RAG pipeline over live PubMed literature, built from scratch across two sessions.

**What was built:**
- A batched PubMed fetcher with structured metadata enrichment via the eSummary API
- An abstract parser with boilerplate cleaning
- A semantic retrieval layer using `all-MiniLM-L6-v2` embeddings and ChromaDB
- A RAG-grounded answer function with inline citations and full references
- An empirical RAG vs. plain LLM comparison on the same query
- An out-of-scope stress test demonstrating RAG's ability to recognise corpus limits

**Key finding:** RAG trades breadth for specificity and provenance. The plain LLM gives the canonical consensus answer; RAG surfaces niche findings (Muse cells, carotid body glomus cells) absent from the LLM response, with every claim traceable to a specific paper. Neither is strictly better — they answer different questions.

This notebook was the development environment for the [PubMed RAG Explorer](https://github.com/Lucia-Cordero/PubMed-RAG) Streamlit app.

# Rationale 

## Why do we need RAG to enhance LLMs?

1. **Recency**. Claude's training data has a cutoff. The papers we pulled include one from June 2026 — after any LLM's training cutoff. No amount of "knowing the field" lets a model cite a paper that didn't exist when it was trained. This is the cleanest, least arguable justification.


> Where RAG wins over plain LLM (with web search mode):

> - **Reproducibility and control** — web search results change over time and aren't a fixed, auditable corpus. RAG over a defined set of papers means the same query gives the same answer every time, which matters for any scientific or regulated context.
> - **Scale and structure** — searching 500+ papers in a structured, queryable way is different from web search returning 10 links. You can do things like "find all abstracts mentioning both KRAS and synthetic lethality" that aren't really a web search use case.
> - **Speed/cost at volume** — if you're going to ask hundreds of questions against the same corpus, embedding once and querying locally is far cheaper and faster than re-searching the web every time.

2. **Verifiability and citation**. Even when an LLM "knows" something about a topic, it can't tell you which specific paper that knowledge came from, and it might be blending or misremembering details across sources (a known failure mode). Our system cites Source 3, Source 5 — meaning you or a reader can go check the actual abstract. That's the difference between "trust me" and "verify me," which matters enormously in a scientific or clinical context.

> Where RAG wins over plain LLM:

> When an LLM cites sources from its own training memory, it can produce plausible-looking but incorrect citations (wrong journal, wrong year, sometimes papers that don't exist — a well-documented failure mode called citation hallucination). When the citation comes from RAG, the source text was actually retrieved and placed in the prompt 

3. **Narrow/private corpora**. This is less true for "stem cells in Parkinson's" specifically, since it's well-represented in any LLM's training data — but imagine instead a corpus of your company's internal unpublished research, a niche subfield with sparse literature, or non-English sources. RAG lets you query data the LLM never saw, which is the dominant real-world use case (e.g. a pharma company doing RAG over their own internal trial data, not public PubMed).
4. **Reducing hallucination on long-tail facts**. LLMs are good at general consensus knowledge but shakier on specific, low-frequency facts — exact effect sizes, specific trial outcomes, rare cell types like Muse cells. Grounding in retrieved text reduces (doesn't eliminate) the chance of confidently wrong specifics.

# Key Findings

## RAG LLM vs plain LLM (Parkinson CT case scenario)
> RAG trades breadth for specificity and provenance!!!

- **Plain Claude is broader and more pedagogically structured** — it draws on the full breadth of its training (which includes textbooks, reviews, consensus knowledge), so it gives you the "canonical" answer a textbook or senior clinician would give. ESCs, iPSCs, NSCs, MSCs, fetal tissue — the well-established five. Notice it even adds general mechanistic context (Yamanaka factors, A9 subtype) that wasn't asked for — that's pattern-completion from broad training, not citation of a specific source.
- **RAG surfaced things plain Claude didn't mention at all** — Muse cells and carotid body glomus cells appear nowhere in the no-RAG answer. That's not a coincidence — these are recent or niche findings, exactly the long-tail, lower-frequency information that's underrepresented in training data and that RAG is specifically good at surfacing.
- **But RAG's answer is narrower and corpus-dependent** — it only knows what's in your 20 abstracts. If your retrieval had missed the iPSC paper, your RAG answer would never have mentioned iPSCs at all, even though they're clearly central to the field. Plain Claude can't make that mistake because it's drawing on broad consensus, not a small fixed set of documents.

## Corpus size & n_results 

When using the larger corpus (n~150), the AI + RAG function did not pick up anymore the Muse cells, carotid... rare cell types. This is because, even though the querying with RAG is deterministic, it is only so at a given point in time and when query/corpus is fixed. Since we expanded the corpus, the n_results  was no longer picking up the rare articles. I tried expanding n_results = 15, but this didn't help. 

The reason increasing n_results to the full corpus doesn't solve it isn't really about compute cost — it's about what cosine similarity actually measures. When you have 19 abstracts, a paper about Muse cells might rank 4th or 5th out of 19 by similarity to your query — close enough to be retrieved. When you have 139 abstracts, that same Muse cells paper might rank 18th or 25th — there are now more papers about mainstream cell types (iPSCs, MSCs, fetal tissue) that are more similar to the query "what cell types are used in stem cell therapy for Parkinson's?" Because Muse cells are niche, their abstract uses different vocabulary and sits further from the query vector than mainstream papers do.
So even retrieving all 139 wouldn't guarantee surfacing it in the most relevant positions — it would just appear somewhere in a long list alongside many less relevant papers, potentially diluting the answer rather than improving it.

# Methodology

## Development process
Session 1
1) Connected to PubMed via an API key to fetch a defined number of articles on a given topic ("Parkinson stem cell therapy")
2) Identified what was the best **chunking strategy** to extract meaningful scientific info: **~100-150 word abstracts** (non-meaningful: title, authors, affiliations)
3) Coded a way to extract "Abstracts" from the rest of the non-meaningful text
4) Removed/cleaned further text that was deemed non-relevant and that sometimes appeared at the end of each abstract (disclosures, competing interests). Used a boilerplate list of words to find and exclude.
5) Embedded abstracts into vectors using an already available transformer: **'all-MiniLM-L6-v2'**. **The choice of transformer is not trivial; they vary in speed and ability to make maintain word semantics**. Here the use of a "science" trained transformer as future improvement could signify a major improvement, although perhaps at the cost of speed, computing power
6) Exploratory: checked the cosine similarity (how close together they are embedded in vector space) of two random abstracts. They were quite different (~0.5-0.6, perfect similarity would be 1.0)! This makes sense based on my knowledge of the Cell Therapy for Parkinson's disease literature.
7) Stored the embeddings (in ChromaDB - specialised database) as a collection.
8) Chose a query (which was itself embedded) : "What cell types are used in stem cell therapy for Parkinson?". Retrieved the most similar chunk(s) in our collection to the chosen query.
9) Packaged retrieved chunks into a prompt and sent it to Claude
10) Included an internal negative control where the prompt (without RAG) was sent to Claude

Session 2
1) Tested the pipeline at larger scale (200 papers instead of 20). Key adjustment was batch processing (n=50)
2) When parsing out abstract chunks, decided to also extract metadata (authors, title, pmids), so as to be able to provide accurate scientific referencing when utilising the AI + RAG feature, which specifically asks for 'sources'
3) Re-embedded the larger pool of abstracts (n~150) and re-stored it in ChromaDB --> new corpus!
4) Included a "negative control" query, purposely out of scope, "How to avoid immune rejection in liver cell therapy"; AI + RAG should acknowledge lack of knowledge, whereas AI will pull on general knowledge to answer.

## Pipeline

**Installing libraries and dependencies**

In [1]:
!pip install requests sentence-transformers chromadb anthropic

import requests
import time
import re
import os
import anthropic
import chromadb
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

load_dotenv()
NCBI_API_KEY = os.getenv("NCBI_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

print(f"NCBI key loaded: {NCBI_API_KEY is not None}")
print(f"Anthropic key loaded: {ANTHROPIC_API_KEY is not None}")

2026-06-25 20:43:34.807843: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-25 20:43:34.808724: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-25 20:43:34.812068: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-25 20:43:34.821295: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-25 20:43:34.838844: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registe

NCBI key loaded: True
Anthropic key loaded: True


**Import functions from utils.py**

In [2]:
from utils import (
    fetch_pubmed_abstracts_batched,
    clean_abstract,
    parse_abstracts_with_metadata,
    fetch_structured_metadata
)

**Fetch PubMed articles (n=200) on Parkinson's and stem cell therapies**

In [3]:
raw_text = fetch_pubmed_abstracts_batched(
    "Stem Cell therapy for Parkinson's disease",
    NCBI_API_KEY,
    max_results=20,
    batch_size=20
)
print(f"Total characters fetched: {len(raw_text)}")

Total characters fetched: 87043


**Extracting paper info (i.e. abstracts + metadata) and cleaning abstracts from noise/unwanted text**

In [4]:
papers = parse_abstracts_with_metadata(raw_text)
print(f"Extracted {len(papers)} papers with metadata\n")

# inspect first result
p = papers[7]
print(f"Title: {p['title']}")
print(f"Authors: {p['authors']}")
print(f"Year: {p['year']}")
print(f"PMID: {p['pmid']}")
print(f"DOI URL: {p['doi_url']}")
print(f"PubMed URL: {p['pubmed_url']}")
print(f"Abstract preview: {p['abstract'][:200]}")

Extracted 19 papers with metadata

Title: Translational stem cell therapy for neurodegeneration and CNS trauma: a focused
Authors: review.
Year: 2026
PMID: 42201594
DOI URL: https://doi.org/10.1007/s11033-026-12011-6
PubMed URL: https://pubmed.ncbi.nlm.nih.gov/42201594/
Abstract preview: Central nervous system (CNS) disorders, including spinal cord injury (SCI) and 
traumatic brain injury (TBI), often result in severe and irreversible functional 
impairments due to complex pathophysio


In [5]:
pmids = [p["pmid"] for p in papers]
structured_metadata = fetch_structured_metadata(pmids, NCBI_API_KEY)

# overwrite imperfect parsed fields with clean API data
for p in papers:
    if p["pmid"] in structured_metadata:
        p["title"] = structured_metadata[p["pmid"]]["title"]
        p["authors"] = structured_metadata[p["pmid"]]["authors"]
        p["year"] = structured_metadata[p["pmid"]]["year"]

# verify first paper
print(f"Title: {papers[0]['title']}")
print(f"Authors: {papers[0]['authors']}")
print(f"Year: {papers[0]['year']}")

Title: A Defined Xeno-Free Matrix Supports Midbrain Dopaminergic Cell Differentiation.
Authors: Wang T, Zeng Y, Xue J, Liu R, Sun J, Bian X, Qiu C, Lan Q, Wu Y, Tang Q, Boyd T, Li L, Schweitzer J, Cheng TL, Chen G, Song B
Year: 2026


**Extracting structured metadata**

**Initialize embedding model**

In [6]:
from sentence_transformers import SentenceTransformer

# load a lightweight but capable embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

**Embed abstracts into vectors**

In [7]:
# embed all abstracts
embeddings = model.encode([p["abstract"] for p in papers], show_progress_bar=True)
print(f"Embedded {len(embeddings)} abstracts")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedded 19 abstracts


**Exploratory cosine similarity between abstracts**

In [8]:
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim_1_2 = cosine_similarity(embeddings[0], embeddings[1])
sim_1_10 = cosine_similarity(embeddings[0], embeddings[9])

print(f"Similarity between abstract 1 and 2: {sim_1_2:.3f}")
print(f"Similarity between abstract 1 and 10: {sim_1_10:.3f}")
print(f"\nAbstract 1 preview: {papers[0]['abstract'][:120]}")
print(f"Abstract 2 preview: {papers[1]['abstract'][:120]}")
print(f"Abstract 10 preview: {papers[9]['abstract'][:120]}")

Similarity between abstract 1 and 2: 0.259
Similarity between abstract 1 and 10: 0.532

Abstract 1 preview: Stem cell-based therapy holds great promise for treating Parkinson's disease 
(PD), yet its clinical translation depends
Abstract 2 preview: The Tau Global Conference 2025, hosted by the Alzheimer's Association, CurePSP, 
and the Rainwater Charitable Foundation
Abstract 10 preview: Parkinson's disease (PD) lacks effective therapeutic methods. Exosomes are 
specialized vesicles that feature a double-l


Scores around 0.5–0.6 indicate related but distinct content — expected, since all abstracts share the same broad topic (stem cell therapy, Parkinson's) but cover different aspects (cell types, mechanisms, clinical outcomes). A score of 1.0 would mean identical vectors. This confirms the embeddings are discriminating between papers rather than collapsing everything into one neighbourhood


**Initialise clients**

In [9]:
# initialise clients
anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
chroma_client = chromadb.Client()

**Storing the embeddings as a collection**

In [10]:
chroma_client = chromadb.Client()

# rebuild collection with metadata
try:
    chroma_client.delete_collection("parkinsons_stem_cell")
except:
    pass

collection = chroma_client.create_collection("parkinsons_stem_cell")

collection.add(
    documents=[p["abstract"] for p in papers],
    embeddings=[e.tolist() for e in embeddings],
    metadatas=[{
        "title": p["title"] or "Unknown title",
        "authors": p["authors"] or "Unknown authors",
        "year": p["year"] or "Unknown year",
        "pubmed_url": p["pubmed_url"],
        "doi_url": p["doi_url"]
    } for p in papers],
    ids=[f"abstract_{i}" for i in range(len(papers))]
)

print(f"Stored {collection.count()} abstracts with metadata")

Stored 19 abstracts with metadata


**Retrieval of 'chunks' from collection with specific query**

In [11]:
query = "What cell types are used in stem cell therapy for Parkinson's?"

In [12]:
query_embedding = model.encode([query])[0].tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

for i, doc in enumerate(results["documents"][0]):
    print(f"--- Result {i+1} ---")
    print(doc[:2000])
    print()

--- Result 1 ---
Stem cell therapy, as a potential treatment for Parkinson's disease (PD), has 
been investigated in many clinical trials for its effectiveness. However, the 
therapeutic outcomes have shown considerable variability. This may be due to 
certain limitations of stem cell therapies, such as the inability to ameliorate 
non-motor symptoms or modify underlying disease mechanisms, and the limited 
applicability in certain patient populations. Combinations with the recently 
emerging gene therapy and other available technologies are expected to make up 
for or break down these limitations of stem cell therapy, thereby broadening its 
scope of application. In this review, we will discuss the limitations of stem 
cell therapy in terms of pathogenesis, symptoms, and population, as well as 
possible coping strategies to enhance the effectiveness of stem cell therapy.

--- Result 2 ---
Neurodegenerative diseases, which afflict millions worldwide and threaten public 
health, have no

**Prompt AI with RAG (by feeding query-retrieved chunks)**

In [13]:
def answer_with_rag(query, collection, model, n_results=5):
    # embed the query
    query_embedding = model.encode([query])[0].tolist()
    
    # retrieve relevant chunks with metadata
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=["documents", "metadatas"]
    )
    
    retrieved_chunks = results["documents"][0]
    metadatas = results["metadatas"][0]
    
    # build context with proper citations
    context = ""
    for i, (chunk, meta) in enumerate(zip(retrieved_chunks, metadatas)):
        context += f"""[Source {i+1}]
Title: {meta['title']}
Authors: {meta['authors']}
Year: {meta['year']}
URL: {meta['pubmed_url']}
Abstract: {chunk}

"""

    system_prompt = """You are a biomedical research assistant. Answer the user's question using ONLY the provided sources.

Rules:
- Synthesise information across all sources where relevant.
- Cite sources by number e.g. (Source 1) inline in your answer.
- At the end of your answer include a References section with full citations in this format:
  [Source N] Authors (Year). Title. URL
- If the sources only partially answer the question, say so explicitly and describe what is missing.
- Do not introduce information not present in the sources, even if you know it from general knowledge."""

    user_prompt = f"""Sources:
{context}

Question: {query}"""

    response = anthropic_client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1500,
        temperature=0,
        system=system_prompt,
        messages=[{"role": "user", "content": user_prompt}]
    )
    
    return response.content[0].text, retrieved_chunks, metadatas

In [14]:
answer, sources, metas = answer_with_rag(
    "What cell types are used in stem cell therapy for Parkinson's?",
    collection,
    model,
    n_results=5
)

In [15]:
print(answer, sources, metas)

## Cell Types Used in Stem Cell Therapy for Parkinson's Disease

Several distinct cell types have been investigated for stem cell-based therapies in Parkinson's disease (PD):

### 1. Fetal Dopaminergic Neurons / Fetal Ventral Mesencephalic Tissue
Early transplantation studies used fetal dopaminergic neurons, which provided the initial proof of concept for cell replacement therapy in PD (Source 3). Transplantation of fetal ventral mesencephalic tissue remains one of the foundational approaches examined in clinical and preclinical work (Source 5).

### 2. Pluripotent Stem Cell-Derived Dopaminergic Progenitors
Induced pluripotent stem cells (iPSCs) can be differentiated into midbrain dopaminergic progenitors, and these are now entering clinical testing (Source 3, Source 5). iPSC-derived exosomes have also been shown to outperform other cell-derived exosomes in therapeutic effects on PD models (Source 4).

### 3. Mesenchymal Stem Cells (MSCs)
MSCs provide neuroprotective and immunomodulato

**Prompt AI without RAG**

In [16]:
def answer_without_rag(query):
    response = anthropic_client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1000,
        temperature=0,
        messages=[{"role": "user", "content": query}]
    )
    return response.content[0].text

query = "What cell types are used in stem cell therapy for Parkinson's?"

plain_answer = answer_without_rag(query)
print(plain_answer)

# Stem Cell Therapy for Parkinson's Disease

Several cell types have been investigated or used in stem cell therapy for Parkinson's disease:

## Primary Cell Types Used

### Embryonic Stem Cells (ESCs)
- Derived from early embryos
- Can be differentiated into **dopaminergic neurons**
- Ethical concerns limit widespread use

### Induced Pluripotent Stem Cells (iPSCs)
- Adult cells reprogrammed to pluripotent state
- Patient-specific (autologous) use possible
- Reduced rejection risk
- Currently a major focus of research

### Fetal Ventral Mesencephalic Tissue
- Contains dopaminergic precursor cells
- Used in early clinical trials (1980s-90s)
- Showed proof-of-concept but inconsistent results
- Ethical and practical limitations

### Neural Stem Cells (NSCs)
- More lineage-restricted
- Can differentiate into neurons and glia

### Mesenchymal Stem Cells (MSCs)
- Derived from bone marrow, adipose tissue
- Primarily **neuroprotective/anti-inflammatory** effects
- Less direct dopamine replace

**Prompt AI (with and without RAG) with out of scope query**

In [17]:
out_of_scope_query = "How can you avoid immune rejection of allogeneic hepatic cells in liver cell therapy?"

# RAG answer
rag_answer, sources, metas = answer_with_rag(out_of_scope_query, collection, model, n_results=5)
print("=== RAG ANSWER ===")
print(rag_answer)
print("\n=== RETRIEVED SOURCES (first 200 chars each) ===")
for i, s in enumerate(sources):
    print(f"\nSource {i+1}: {s[:200]}")

# Plain Claude answer
plain_answer = answer_without_rag(out_of_scope_query)
print("\n=== PLAIN CLAUDE ANSWER ===")
print(plain_answer)

=== RAG ANSWER ===
The provided sources do not contain information relevant to this question. All five sources focus exclusively on **Parkinson's disease** treatments, covering topics such as stem cell therapy, Muse cells, mesenchymal stromal cells, and exosome-based approaches for neurological conditions. None of the sources address:

- Liver cell therapy or hepatic cell transplantation
- Immune rejection mechanisms in the context of allogeneic cell transplantation
- Immunosuppression strategies for liver-directed therapies
- Tolerance induction for hepatic grafts

Therefore, I am unable to answer the question about avoiding immune rejection of allogeneic hepatic cells in liver cell therapy using only the provided sources. To answer this question accurately, sources specifically addressing **hepatocyte transplantation**, **liver-directed gene/cell therapy**, or **transplant immunology in hepatic contexts** would be required.

---

**References:**

[Source 1] Li Z, Peng S, Ouyang J, Li